In [52]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo

Persona generation (sample of 10)

In [41]:
df = pd.read_csv("Synthetic_Big5_data.csv")
df = df.round(2)
df = df.sample(n=10, random_state=1)
df

,Enthusiasm,Assertiveness,Compassion,Politeness,Industriousness,Orderliness,Volatility,Withdrawal,Intellect,Openness,gender,Extraversion,Agreeableness,Conscientiousness,Neuroticism,Openness_Intellect
507,3.89,2.96,4.24,4.10,3.21,3.64,1.92,3.22,1.91,2.95,female,3.42,4.17,3.43,2.57,2.43
818,5.28,4.09,4.77,4.33,2.65,3.53,2.23,2.90,3.10,3.10,female,4.68,4.55,3.09,2.57,3.10
452,3.37,3.15,4.27,4.44,2.53,3.37,2.17,2.93,3.01,3.00,male,3.26,4.35,2.95,2.55,3.01
368,2.75,3.43,3.64,3.56,2.70,2.94,2.66,2.90,4.28,4.52,male,3.09,3.60,2.82,2.78,4.40
242,5.05,4.06,3.85,3.25,2.98,3.26,2.00,1.83,3.72,3.73,male,4.56,3.55,3.12,1.91,3.72
929,3.85,2.96,3.43,2.56,2.80,2.92,3.62,3.03,2.66,2.96,female,3.41,2.99,2.86,3.32,2.81
262,2.96,3.74,3.68,3.32,2.35,3.21,2.39,3.07,3.35,3.81,male,3.35,3.50,2.78,2.73,3.58
810,3.53,3.04,4.60,3.75,2.69,3.13,3.08,3.71,3.18,4.49,female,3.29,4.18,2.91,3.39,3.84
318,3.02,3.01,4.00,3.59,3.11,3.30,2.76,2.94,2.81,2.96,male,3.02,3.79,3.20,2.85,2.88
49,4.64,3.09,3.91,2.94,3.23,2.96,2.93,2.72,3.84,3.14,male,3.87,3.43,3.10,2.82,3.49


In [ ]:
PERSONA_SYSTEM_PROMPT = """
    You are a professional personality analyst and storyteller. Your task is to generate a **realistic, detailed personality profile** of a single individual based on their **Big Five subfacet scores on a 1-5 Likert scale**.

    The subfacets are:

    - Extraversion: Enthusiasm, Assertiveness
    - Agreeableness: Compassion, Politeness
    - Conscientiousness: Industriousness, Orderliness
    - Neuroticism: Volatility, Withdrawal
    - Openness to Experience: Intellect, Openness

    Instructions:

    1. Input will provide numeric scores (1-5) for each subfacet.  

    2. Output should be a **detailed personality description (~800 words)** that represents a **coherent, realistic individual** consistent with the scores.  

    3. Use the subfacet scores to shape the description:
    - High scores (4-5) → emphasize frequent, strong expression of the trait  
    - Medium scores (~3) → portray moderate, balanced expression  
    - Low scores (1-2) → portray subtle or infrequent expression  

    4. Include, organically in the narrative:
    - Behavior patterns in social, work, and personal contexts  
    - Typical emotional reactions and coping styles  
    - Interaction style with others, including assertiveness, empathy, and politeness  
    - Cognitive tendencies, interests, and openness to new experiences  
    - Habits, routines, and level of conscientiousness  

    5. The output should read like a **psychologically coherent personality profile**, not just a list of traits. Do not include raw scores in the text.

    6. The text should be suitable to be used as a **persona description** for an LLM that will later simulate the behavior of this individual.

    7. The tone should be **formal but narrative**, suitable for psychology research, character modeling, or behavioral simulation.

    8. Always produce a **single continuous text**, approximately 800 words.

    In the next message you will recieve the personality trait messages:
""" 

In [ ]:
def call_llm(llm_model, sys_prompt, message):
    
    if llm_model == "deepseek":
        API_KEY = "DEEPSEEK_API_KEY"
        MODEL_NAME = "deepseek-chat"
        MODEL_URL = "https://api.deepseek.com"
        
    elif llm_model == "qwen":
        # ADJUST THIS LATER
        API_KEY = "DEEPSEEK_API_KEY"
        MODEL_NAME = "deepseek-chat"
        MODEL_URL = "https://api.deepseek.com"
        
    elif llm_model == "llama":
        # ADJUST THIS LATER
        API_KEY = "DEEPSEEK_API_KEY"
        MODEL_NAME = "deepseek-chat"
        MODEL_URL = "https://api.deepseek.com"
        
    load_dotenv() 
    client = OpenAI(
        api_key=os.environ.get(API_KEY),
        base_url=MODEL_URL
        )

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": message},
        ],
        stream=False
    )
    date = response.created # THIS NEEDS FIXING (i think its sec after 1970)
    date = datetime.fromtimestamp(
            response.created,
            tz=ZoneInfo("Europe/Vienna")
        )
    response_text = response.choices[0].message.content
    
    return response_text, date

responses = []
dates = []
model = "deepseek"
for _, row in df.iterrows():
    message = "\n".join(f"{col}: {row[col]}" for col in df.columns)
    response_text, date = call_llm(model, PERSONA_SYSTEM_PROMPT, message)
    
    responses.append(response_text)
    dates.append(date)

df["llm_persona"] = responses 
df["persona_generation_date"] = dates
df["llm_persona_generation"] = model

In [ ]:
# df.to_csv("sample_persona.csv", index=False)

Experiment Setup:

In [ ]:
def run_experiment(llm_model, instructions):

    for experiment_dict in instructions:
        for experiment_name, experiment in experiment_dict.items():

            responses_experiment = []
            date_experiment = []

            for _, row in df.iterrows():
                PERSONA_SYSTEM_PROMPT = f"""
                Your task is to behave in the way that the Personality Profile describes:
                
                {row["llm_persona"]}
                
                and respond in this format....

                {experiment.sys_instruction}
                """

                response_text, date = call_llm(
                    llm_model,
                    PERSONA_SYSTEM_PROMPT,
                    experiment.prompt
                )

                responses_experiment.append(response_text)
                date_experiment.append(date)

            df[f"{llm_model}_experiment_{experiment_name}"] = responses_experiment
            df[f"{llm_model}_experiment_{experiment_name}_date"] = date_experiment
            return

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True) # make it immutable
class Experiment:
    prompt: str
    sys_instruction: str

instructions = [
    {
        "Gamblers_Fallacy_1": Experiment(
            prompt="Experiments Text to LLM",
            sys_instruction="ADDITIONAL SYS PROMPT INSTRUCTION"
        )
    },
    {
        "Gamblers_Fallacy_2": Experiment(
            prompt="Experiments Text to LLM",
            sys_instruction="ADDITIONAL SYS PROMPT INSTRUCTION"
        )
    }
]


run_experiment("deepseek", instructions) 